Implentacion de la libreria [Anomalib](https://anomalib.readthedocs.io/en/latest/index.html)

In [1]:
from anomalib.data import MVTecAD
from anomalib.deploy import ExportType
from anomalib.engine import Engine
from anomalib.models import Patchcore
from anomalib.visualization import ImageVisualizer

pre_processor = Patchcore.configure_pre_processor(image_size=(192, 192))

datamodule = MVTecAD(
    root="/home/jamor/redes/MVTec-AD/data/raw/",  # Path to download/store the dataset
    category="screw",  # MVTec category to use
    train_batch_size=2,  # Number of images per training batch
    eval_batch_size=2,  # Number of images per validation/test batch
)

visualizer = ImageVisualizer(
    overlay_fields=[
         ("image", ["anomaly_map"]),
         ("image", ["pred_mask"])
     ],
    fields_config={"anomaly_map": {"normalize": True}},
    output_dir="/home/jamor/redes/MVTec-AD/reports/anomalib",
    text_config={"size":8}
)

model = Patchcore(
    num_neighbors=6,  # Override default model settings
    pre_processor=pre_processor,
    coreset_sampling_ratio=0.01,
    visualizer=visualizer
)

engine = Engine(
    accelerator="gpu",
    precision="bf16-mixed",   # Halves VRAM usage — most important setting
    devices=1,
    default_root_dir="/home/jamor/redes/MVTec-AD/models/anomalib"
)

/home/jamor/redes/MVTec-AD/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'pre_processor' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['pre_processor'])`.


In [2]:
import sys
import rich.jupyter
rich.jupyter.JUPYTER_HTML_FORMAT = ""
from unittest.mock import patch
import anomalib.models.components.sampling.k_center_greedy as kcg
from tqdm import tqdm as original_tqdm
kcg.tqdm = lambda iterable, **kwargs: iterable
# to avoid recursion error due to tqm
engine.fit(datamodule=datamodule, model=model)

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 3050 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/jamor/redes/MVTec-AD/.venv/lib/python3.12/site-packages/lightning/pytorch/core/optimizer.py:183: `LightningModule.configure_optimizers` returned `None`, this fit will run with no optimizer
/home/jamor/redes/MVTec-AD/.ven

┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ pre_processor  │ PreProcessor   │      0 │ train │     0 │
│ 1 │ post_processor │ PostProcessor  │      0 │ train │     0 │
│ 2 │ evaluator      │ Evaluator      │      0 │ train │     0 │
│ 3 │ model          │ PatchcoreModel │ 24.9 M │ train │     0 │
└───┴────────────────┴────────────────┴────────┴───────┴───────┘

Trainable params: 24.9 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.9 M                                                                                               
Total estimated model params size (MB): 99                                                                         
Modules in train mode: 19                                                                                          
Modules in eval mode: 174                                                                                          
Total FLOPs: 0

/home/jamor/redes/MVTec-AD/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

/home/jamor/redes/MVTec-AD/.venv/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:534: Found 174 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


`Trainer.fit` stopped: `max_epochs=1` reached.


In [3]:
test_results = engine.test(datamodule=datamodule, model=model)
test_results

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │    0.9040787220001221     │
│       image_F1Score       │    0.9053497910499573     │
│        pixel_AUROC        │    0.9505389332771301     │
│       pixel_F1Score       │    0.2824091911315918     │
└───────────────────────────┴───────────────────────────┘

[{'image_AUROC': 0.9040787220001221,
  'image_F1Score': 0.9053497910499573,
  'pixel_AUROC': 0.9505389332771301,
  'pixel_F1Score': 0.2824091911315918}]